<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/Type_3_using_Shared_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numba
# Install pre-compiled CuPy wheel for CUDA 12.x to avoid building from source (takes ~15s instead of 15+ minutes)
!pip install cupy-cuda12x

In [15]:
%%writefile matmul-03.py
from numba import cuda , float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b -1) // b
tile_dim = 32

@cuda.jit
def matmul(matA, matB, matC, m, n, k):
  tileA = cuda.shared.array(shape=(tile_dim,tile_dim), dtype = cp.float32)
  tileB = cuda.shared.array(shape=(tile_dim,tile_dim), dtype = cp.float32)

  tx = cuda.threadIdx.x
  ty = cuda.threadIdx.y

  out_col = tx + cuda.blockIdx.x * cuda.blockDim.x
  out_row = ty + cuda.blockIdx.y * cuda.blockDim.y

  num_tiles = (k + tile_dim - 1) // tile_dim

  acc = float32(0.0)
  for t in range(num_tiles):
    # phase 1:
    # threads within the block
    # will read elements from matA and matB (gmem)
    # and store them in tileA and tileB (smem)
    tile_offset = t * tile_dim
    a_col = tx + tile_offset
    b_row = ty + tile_offset

    if out_row < m and a_col < k:
      tileA[ty,tx] = matA[out_row, a_col]
    else:
      tileA[ty,tx] = float32(0.0)

    if out_col < n and b_row < k:
      tileB[ty,tx] = matB[b_row, out_col]
    else:
      tileB[ty,tx] = float32(0.0)

    cuda.syncthreads()

    # phase 2:
    # threads will compute dot product
    # by reading elements from tileA and tileB (smem)
    for i in range(tile_dim):
        acc += tileA[ty,i] * tileB[i,tx]

    cuda.syncthreads()

  if out_row < m and out_col < n:
    matC[out_row, out_col] = acc


m = 1000
k = 1200
n = 800

matA = cp.ones(shape=(m,k), dtype=cp.float32) * 2.0
matB = cp.ones(shape=(k,n), dtype=cp.float32) * 3.0
# Fixed typo: changed cp.zero to cp.zeros
matC = cp.zeros(shape=(m,n), dtype=cp.float32)

threads_per_block = (32,32)
blocks_per_grid = (ceil_div(n, threads_per_block[0]), ceil_div(m, threads_per_block[1]))
matmul[blocks_per_grid,threads_per_block](matA, matB, matC, m, n, k)

cuda.synchronize()

matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
# Fixed typo: changed cp.asumpy to cp.asnumpy
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5, atol=1e-5)

Writing matmul-03.py


In [17]:
!ncu --set full -f -o matmul-03.ncu-rep python3 matmul-03.py

/usr/local/lib/python3.13/dist-packages/cupy/_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''
==PROF== Connected to process 47340 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 50 passes
